# 🤖 BudgetIA PME — Entraînement Module 1
## Classification Automatique des Transactions Financières (OHADA)

**Hackathon IA & Cybersécurité 2026 — ISM Dakar — Groupe 17**

Auteurs : Branham NDZIENGOMO DJAGNI · ISSEMIBA Edwine Roxane

---

### Ce que fait ce notebook :
1. Installe les bibliothèques nécessaires
2. Génère le dataset de 612 transactions PME sénégalaises
3. Entraîne le classifieur **TF-IDF + Random Forest** (catégories OHADA)
4. Entraîne le détecteur d'anomalies **Isolation Forest**
5. Affiche les performances et des exemples de prédiction
6. Télécharge les modèles `.pkl` prêts à utiliser

## ⚙️ Étape 1 — Installation des bibliothèques

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet
print('✅ Bibliothèques installées')

## 📊 Étape 2 — Génération du Dataset PME Sénégalaise

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

# ── Catégories OHADA ─────────────────────────────────────────────────────
CATEGORIES = {
    '601 - Achats de marchandises':        ('charge', ['ACHAT STOCK', 'FOURNISSEUR', 'MARCHANDISES', 'APPROVISIONNEMENT', 'ACHAT PRODUITS']),
    '621 - Salaires et traitements':       ('charge', ['SALAIRES', 'PAIE', 'REMUNERATION', 'TRAITEMENT MENSUEL', 'SALAIRES EMPLOYES']),
    '624 - Transport et déplacements':     ('charge', ['TAXI', 'TRANSPORT', 'LIVRAISON', 'DEPLACEMENT', 'CARBURANT']),
    '626 - Télécommunications':            ('charge', ['FACTURE ORANGE', 'ABONNEMENT INTERNET', 'FACTURE EXPRESSO', 'FACTURE FREE', 'MOBILE']),
    '627 - Publicité et communication':    ('charge', ['PUBLICITE', 'FACEBOOK ADS', 'IMPRESSION', 'AFFICHAGE', 'MARKETING']),
    '631 - Loyer local commercial':        ('charge', ['LOYER BOUTIQUE', 'LOYER LOCAL', 'LOYER MAGASIN', 'LOYER BUREAU', 'LOYER PLATEAU']),
    '641 - Eau et électricité':            ('charge', ['FACTURE SENELEC', 'ELECTRICITE', 'FACTURE SDE', 'EAU DAKAR', 'SENELEC']),
    '658 - Charges diverses':              ('charge', ['FRAIS DIVERS', 'FOURNITURES BUREAU', 'CHARGES DIVERSES', 'DEPENSES DIVERSES', 'FRAIS BANCAIRES']),
    '701 - Ventes de marchandises':        ('produit', ['RECETTE JOURNALIERE', 'VENTE CAISSE', 'RECETTE CAISSE', 'VENTES JOURNEE', 'CHIFFRE AFFAIRES']),
    '706 - Prestations de services':       ('produit', ['VIREMENT CLIENT', 'PAIEMENT CLIENT', 'HONORAIRES', 'PRESTATION', 'VIREMENT RECU']),
}

# ── Saisonnalité sénégalaise 2025 ────────────────────────────────────────
def multiplicateur_saisonnier(d):
    mois, jour = d.month, d.day
    # Tabaski (Juin)
    if mois == 6 and 5 <= jour <= 10:   return 2.1
    # Korité (Mars)
    if mois == 3 and 28 <= jour <= 31:  return 1.9
    # Ramadan (Mars-Avril)
    if (mois == 3 and jour >= 1) or (mois == 4 and jour <= 20): return 1.4
    # Tamkharit (Juillet)
    if mois == 7 and 25 <= jour <= 27:  return 1.2
    # Rentrée scolaire (Octobre)
    if mois == 10 and 1 <= jour <= 15:  return 1.35
    # Fêtes de fin d'année (Décembre)
    if mois == 12 and jour >= 20:       return 1.55
    return 1.0

# ── Génération des transactions ──────────────────────────────────────────
transactions = []
debut = date(2025, 1, 1)
fin   = date(2025, 12, 31)
tid   = 1

d = debut
while d <= fin:
    jour_semaine = d.weekday()  # 0=Lundi, 6=Dimanche
    coef = multiplicateur_saisonnier(d)

    # Charges fixes le 1er du mois
    if d.day == 1:
        fixes = [
            ('631 - Loyer local commercial',    f'LOYER BOUTIQUE PLATEAU DAKAR {d.strftime("%B").upper()} {d.year}', 150000),
            ('621 - Salaires et traitements',   f'SALAIRES EMPLOYES {d.strftime("%B").upper()} {d.year}',           int(random.gauss(300000, 30000))),
            ('641 - Eau et électricité',         f'FACTURE SENELEC ELECTRICITE {d.strftime("%B").upper()}',          int(random.gauss(45000, 8000))),
            ('626 - Télécommunications',         f'FACTURE ORANGE MOBILE INTERNET {d.strftime("%B").upper()}',       int(random.gauss(25000, 3000))),
        ]
        for cat, lib, mnt in fixes:
            transactions.append({'id': tid, 'date': d.isoformat(), 'libelle': lib,
                                  'categorie': cat, 'type': CATEGORIES[cat][0],
                                  'montant': max(mnt, 5000), 'anomalie': False})
            tid += 1

    # Achats fournisseurs (Lun, Mer, Ven)
    if jour_semaine in [0, 2, 4]:
        mnt = int(random.gauss(180000, 40000) * coef)
        lib = random.choice(CATEGORIES['601 - Achats de marchandises'][1])
        lib += f' {random.choice(["SANDAGA", "TILENE", "HLM", "COLOBANE", "MARCHE KERMEL"])}'
        transactions.append({'id': tid, 'date': d.isoformat(), 'libelle': lib,
                              'categorie': '601 - Achats de marchandises', 'type': 'charge',
                              'montant': max(mnt, 10000), 'anomalie': False})
        tid += 1

    # Ventes (tous les jours sauf Dimanche)
    if jour_semaine != 6:
        mnt = int(random.gauss(75000, 20000) * coef)
        lib = random.choice(CATEGORIES['701 - Ventes de marchandises'][1])
        lib += f' {d.strftime("%A").upper()} {d.day}/{d.month}'
        transactions.append({'id': tid, 'date': d.isoformat(), 'libelle': lib,
                              'categorie': '701 - Ventes de marchandises', 'type': 'produit',
                              'montant': max(mnt, 5000), 'anomalie': False})
        tid += 1

    # Autres charges aléatoires (2x par semaine)
    if jour_semaine in [1, 3] and random.random() > 0.4:
        autres_cats = ['624 - Transport et déplacements', '627 - Publicité et communication',
                       '658 - Charges diverses', '706 - Prestations de services']
        cat = random.choice(autres_cats)
        lib = random.choice(CATEGORIES[cat][1]) + f' {d.strftime("%B").upper()}'
        mnt = int(random.gauss(35000, 12000))
        transactions.append({'id': tid, 'date': d.isoformat(), 'libelle': lib,
                              'categorie': cat, 'type': CATEGORIES[cat][0],
                              'montant': max(mnt, 2000), 'anomalie': False})
        tid += 1

    d += timedelta(days=1)

# ── 5 Anomalies injectées ────────────────────────────────────────────────
anomalies = [
    {'id': tid,   'date': '2025-04-15', 'libelle': 'VIREMENT SUSPECT COMPTE INCONNU REF 4421',
     'categorie': '658 - Charges diverses', 'type': 'charge', 'montant': 2500000, 'anomalie': True},
    {'id': tid+1, 'date': '2025-07-03', 'libelle': 'DOUBLE PAIEMENT LOYER BOUTIQUE JUILLET',
     'categorie': '631 - Loyer local commercial', 'type': 'charge', 'montant': 300000, 'anomalie': True},
    {'id': tid+2, 'date': '2025-09-20', 'libelle': 'ACHAT FOURNISSEUR NON HABITUEL SANS BON',
     'categorie': '601 - Achats de marchandises', 'type': 'charge', 'montant': 890000, 'anomalie': True},
    {'id': tid+3, 'date': '2025-11-08', 'libelle': 'RETRAIT DAB DIMANCHE MINUIT ECOBANK',
     'categorie': '658 - Charges diverses', 'type': 'charge', 'montant': 450000, 'anomalie': True},
    {'id': tid+4, 'date': '2025-12-01', 'libelle': 'VIREMENT ETRANGER NON IDENTIFIE SWIFT',
     'categorie': '658 - Charges diverses', 'type': 'charge', 'montant': 1200000, 'anomalie': True},
]
transactions.extend(anomalies)

df = pd.DataFrame(transactions)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.to_csv('transactions_pme.csv', index=False)

print(f'✅ Dataset généré : {len(df)} transactions')
print(f'   Période : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'   Anomalies : {df["anomalie"].sum()}')
print()
print('Répartition par catégorie :')
print(df['categorie'].value_counts().to_string())

## 🔍 Étape 3 — Exploration du Dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)

df = pd.read_csv('transactions_pme.csv')
df['date'] = pd.to_datetime(df['date'])

print('=== APERÇU DU DATASET ===')
display(df.head(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Répartition des catégories
counts = df['categorie'].value_counts()
axes[0].barh(counts.index, counts.values, color='steelblue')
axes[0].set_title('Nombre de transactions par catégorie OHADA')
axes[0].set_xlabel('Nombre')

# Montants par mois
df['mois'] = df['date'].dt.month
ventes = df[df['type']=='produit'].groupby('mois')['montant'].sum()
charges = df[df['type']=='charge'].groupby('mois')['montant'].sum()
axes[1].plot(ventes.index, ventes.values, 'g-o', label='Produits')
axes[1].plot(charges.index, charges.values, 'r-o', label='Charges')
axes[1].set_title('Évolution mensuelle Produits vs Charges (FCFA)')
axes[1].set_xlabel('Mois')
axes[1].legend()
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 🤖 Étape 4 — Entraînement du Classifieur TF-IDF + Random Forest

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pickle

df = pd.read_csv('transactions_pme.csv')

X = df['libelle'].values
y = df['categorie'].values

# Découpage train/test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Données entraînement : {len(X_train)} transactions')
print(f'Données test         : {len(X_test)} transactions')
print()

# Pipeline TF-IDF + Random Forest
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),   # mots simples + paires de mots
        max_features=5000,    # garder les 5000 mots les plus importants
        analyzer='word',
    )),
    ('rf', RandomForestClassifier(
        n_estimators=200,          # 200 arbres de décision
        class_weight='balanced',   # équilibrer les catégories
        random_state=42,
        n_jobs=-1,                 # utiliser tous les CPU disponibles
    )),
])

print('⏳ Entraînement en cours...')
pipeline.fit(X_train, y_train)
print('✅ Entraînement terminé !')
print()

# Évaluation
y_pred = pipeline.predict(X_test)
precision = accuracy_score(y_test, y_pred)
print(f'🎯 PRÉCISION GLOBALE : {precision*100:.1f}%')
print()
print('=== RAPPORT DÉTAILLÉ PAR CATÉGORIE ===')
print(classification_report(y_test, y_pred))

# Sauvegarde
with open('classifieur_categories.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print('✅ Modèle sauvegardé : classifieur_categories.pkl')

## 🧪 Étape 5 — Tester le Classifieur

In [ ]:
def classifier_transaction(libelle):
    probas   = pipeline.predict_proba([libelle])[0]
    classes  = pipeline.classes_
    idx      = probas.argmax()
    top3_idx = probas.argsort()[::-1][:3]
    return {
        'libelle':   libelle,
        'categorie': classes[idx],
        'confiance': f'{probas[idx]*100:.1f}%',
        'top3': [(classes[i], f'{probas[i]*100:.1f}%') for i in top3_idx]
    }

tests = [
    'LOYER BOUTIQUE PLATEAU DAKAR',
    'RECETTE JOURNALIERE CAISSE VENDREDI',
    'FACTURE SENELEC ELECTRICITE MARS',
    'SALAIRES EMPLOYES FEVRIER 2025',
    'PUBLICITE FACEBOOK ADS CAMPAGNE',
    'TAXI LIVRAISON CLIENT MERMOZ',
    'FACTURE ORANGE MOBILE INTERNET',
    'VIREMENT CLIENT DIALLO FILS',
]

print('=== RÉSULTATS DE CLASSIFICATION ===')
print(f'{"LIBELLÉ":<45} {"CATÉGORIE":<35} {"CONFIANCE"}')
print('-' * 90)
for libelle in tests:
    r = classifier_transaction(libelle)
    print(f'{r["libelle"]:<45} {r["categorie"]:<35} {r["confiance"]}')

## 🚨 Étape 6 — Entraînement de l'Isolation Forest (Détection d'Anomalies)

In [ ]:
from sklearn.ensemble import IsolationForest

df = pd.read_csv('transactions_pme.csv')
df['date'] = pd.to_datetime(df['date'])
df['mois']         = df['date'].dt.month
df['jour_semaine'] = df['date'].dt.dayofweek
df['est_charge']   = (df['type'] == 'charge').astype(int)

features = df[['montant', 'mois', 'jour_semaine', 'est_charge']].values

iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,   # on estime 2% de transactions anormales
    random_state=42,
)

print('⏳ Entraînement Isolation Forest...')
df['score_anomalie'] = iso.fit_predict(features)
df['est_anomalie']   = df['score_anomalie'] == -1
print('✅ Terminé !')
print()

print('=== ANOMALIES DÉTECTÉES ===')
anomalies_detectees = df[df['est_anomalie'] == True][['date', 'libelle', 'montant', 'anomalie']]
display(anomalies_detectees)

# Sauvegarde
with open('isolation_forest.pkl', 'wb') as f:
    pickle.dump(iso, f)
print()
print('✅ Modèle sauvegardé : isolation_forest.pkl')

## 📊 Étape 7 — Visualisation des Anomalies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution des montants
normales  = df[df['est_anomalie'] == False]['montant']
suspectes = df[df['est_anomalie'] == True]['montant']

axes[0].hist(normales,  bins=40, color='steelblue', alpha=0.7, label='Normales')
axes[0].hist(suspectes, bins=10, color='red',       alpha=0.9, label='Suspectes')
axes[0].set_title('Distribution des montants — Normales vs Suspectes')
axes[0].set_xlabel('Montant (FCFA)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Montant vs Mois (scatter)
colors = ['red' if a else 'steelblue' for a in df['est_anomalie']]
axes[1].scatter(df['mois'], df['montant'], c=colors, alpha=0.6, s=30)
axes[1].set_title('Montant par mois — Anomalies en rouge')
axes[1].set_xlabel('Mois')
axes[1].set_ylabel('Montant (FCFA)')
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 💾 Étape 8 — Télécharger les Modèles

In [ ]:
import os

fichiers = [
    'transactions_pme.csv',
    'classifieur_categories.pkl',
    'isolation_forest.pkl',
]

print('Fichiers disponibles :')
for f in fichiers:
    taille = os.path.getsize(f) / 1024
    print(f'  {f:<35} {taille:.1f} Ko')

# Téléchargement (uniquement sur Google Colab)
try:
    from google.colab import files
    print()
    print('⬇️  Téléchargement des fichiers...')
    for f in fichiers:
        files.download(f)
        print(f'  ✅ {f} téléchargé')
except ImportError:
    print()
    print('(Téléchargement automatique uniquement sur Google Colab)')
    print('Les fichiers sont dans le dossier courant.')

## ✅ Récapitulatif

| Fichier généré | Description |
|---|---|
| `transactions_pme.csv` | Dataset de 612 transactions PME sénégalaises |
| `classifieur_categories.pkl` | Modèle TF-IDF + Random Forest entraîné |
| `isolation_forest.pkl` | Modèle Isolation Forest entraîné |

### Pour utiliser les modèles dans l'app Streamlit :
Placer les `.pkl` dans le dossier `models/` du projet BudgetIA PME.

---
*Hackathon IA & Cybersécurité 2026 — ISM Dakar — Groupe 17*